## Montar Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = '/content/drive/MyDrive/Meli_Case'

In [3]:
import sys
sys.path.append(path)

## Imports

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
from scipy.stats import ks_2samp
import shap
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler

## Funções

In [6]:
from functions import calculate_psi, analisar_metricas_por_decil

## Base

In [7]:
base = pd.read_excel(f"{path}/fraud_dataset.xlsx")
base = base.rename(columns={col: f"var_{col}" for col in base.columns if len(col) == 1 and col.isalpha()})

In [8]:
variaveis = ["var_Q", "var_R", "var_S", 'Monto']

for col in variaveis:
    base[col] = base[col].apply(lambda x: np.nan if isinstance(x, datetime) else x)

    base[col] = base[col].astype(str).str.replace(",", "").str.replace(" ", "").replace("None", np.nan)
    base[col] = pd.to_numeric(base[col], errors='coerce')

In [9]:
for col in ["var_C", "var_F", "var_G"]:
    base[f"{col}_log"] = np.log1p(base[col])

In [10]:
def mapear_pais(pais):
    if pais in ['BR', 'AR', 'MX']:
        return pais
    elif pais in ['ES', 'US', 'UY']:
        return 'ES_US_UY'
    else:
        return 'OUTROS'

base['pais_agg'] = base['var_J'].apply(mapear_pais)

In [11]:
pais_agg_dummies = pd.get_dummies(base['pais_agg'], prefix='pais_agg')
pais_agg_dummies = pais_agg_dummies.astype(int)

In [12]:
base = pd.concat([base, pais_agg_dummies], axis=1)
base

,var_A,var_B,var_C,var_D,var_E,var_F,var_G,var_H,var_I,var_J,...,Fraude,var_C_log,var_F_log,var_G_log,pais_agg,pais_agg_AR,pais_agg_BR,pais_agg_ES_US_UY,pais_agg_MX,pais_agg_OUTROS
0,0,10,50257.0,0,0,0.0,0.0,0,0,UY,...,1,10.824925,0.000000,0.0,ES_US_UY,0,0,1,0,0
1,0,10,29014.0,0,0,0.0,0.0,0,0,UY,...,1,10.275568,0.000000,0.0,ES_US_UY,0,0,1,0,0
2,0,7,92.0,0,1,0.0,0.0,0,1,UY,...,1,4.532599,0.000000,0.0,ES_US_UY,0,0,1,0,0
3,9,16,50269.0,0,0,0.0,0.0,0,0,UY,...,1,10.825164,0.000000,0.0,ES_US_UY,0,0,1,0,0
4,0,8,8180.0,0,0,0.0,0.0,0,0,UY,...,1,9.009570,0.000000,0.0,ES_US_UY,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16875,0,3,63302.0,0,1,0.5,0.0,0,0,BR,...,1,11.055688,0.405465,0.0,BR,0,1,0,0,0
16876,0,12,825.0,0,0,0.0,0.0,0,0,BR,...,1,6.716595,0.000000,0.0,BR,0,1,0,0,0
16877,1,3,81067.0,0,0,0.0,0.0,0,0,BR,...,1,11.303044,0.000000,0.0,BR,0,1,0,0,0
16878,0,9,398372.0,0,0,0.0,0.0,0,0,BR,...,1,12.895144,0.000000,0.0,BR,0,1,0,0,0


In [13]:
base['var_K_100'] = (base['var_K'] * 100).round()

In [14]:
drop_col = ['var_C', 'var_F', 'var_G', 'var_J', 'pais_agg', 'var_K']
base = base.drop(columns=drop_col, errors='ignore')
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16880 entries, 0 to 16879
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   var_A              16880 non-null  int64  
 1   var_B              16880 non-null  int64  
 2   var_D              16880 non-null  int64  
 3   var_E              16880 non-null  int64  
 4   var_H              16880 non-null  int64  
 5   var_I              16880 non-null  int64  
 6   var_L              16880 non-null  int64  
 7   var_M              16880 non-null  int64  
 8   var_N              16880 non-null  int64  
 9   var_O              16880 non-null  int64  
 10  var_P              16880 non-null  int64  
 11  var_Q              16863 non-null  float64
 12  var_R              16877 non-null  float64
 13  var_S              15575 non-null  float64
 14  Monto              16413 non-null  float64
 15  Fraude             16880 non-null  int64  
 16  var_C_log          136

In [15]:
df = base[base['Monto'].notna()]

## Base

In [16]:
base = pd.read_excel(f"{path}/fraud_dataset.xlsx")
base = base.rename(columns={col: f"var_{col}" for col in base.columns if len(col) == 1 and col.isalpha()})

In [17]:
variaveis = ["var_Q", "var_R", "var_S", 'Monto']

for col in variaveis:
    base[col] = base[col].apply(lambda x: np.nan if isinstance(x, datetime) else x)

    base[col] = base[col].astype(str).str.replace(",", "").str.replace(" ", "").replace("None", np.nan)
    base[col] = pd.to_numeric(base[col], errors='coerce')

In [18]:
for col in ["var_C", "var_F", "var_G"]:
    base[f"{col}_log"] = np.log1p(base[col])

In [19]:
def mapear_pais(pais):
    if pais in ['BR', 'AR', 'MX']:
        return pais
    elif pais in ['ES', 'US', 'UY']:
        return 'ES_US_UY'
    else:
        return 'OUTROS'

base['pais_agg'] = base['var_J'].apply(mapear_pais)

In [20]:
pais_agg_dummies = pd.get_dummies(base['pais_agg'], prefix='pais_agg')
pais_agg_dummies = pais_agg_dummies.astype(int)

In [21]:
base = pd.concat([base, pais_agg_dummies], axis=1)
base

,var_A,var_B,var_C,var_D,var_E,var_F,var_G,var_H,var_I,var_J,...,Fraude,var_C_log,var_F_log,var_G_log,pais_agg,pais_agg_AR,pais_agg_BR,pais_agg_ES_US_UY,pais_agg_MX,pais_agg_OUTROS
0,0,10,50257.0,0,0,0.0,0.0,0,0,UY,...,1,10.824925,0.000000,0.0,ES_US_UY,0,0,1,0,0
1,0,10,29014.0,0,0,0.0,0.0,0,0,UY,...,1,10.275568,0.000000,0.0,ES_US_UY,0,0,1,0,0
2,0,7,92.0,0,1,0.0,0.0,0,1,UY,...,1,4.532599,0.000000,0.0,ES_US_UY,0,0,1,0,0
3,9,16,50269.0,0,0,0.0,0.0,0,0,UY,...,1,10.825164,0.000000,0.0,ES_US_UY,0,0,1,0,0
4,0,8,8180.0,0,0,0.0,0.0,0,0,UY,...,1,9.009570,0.000000,0.0,ES_US_UY,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16875,0,3,63302.0,0,1,0.5,0.0,0,0,BR,...,1,11.055688,0.405465,0.0,BR,0,1,0,0,0
16876,0,12,825.0,0,0,0.0,0.0,0,0,BR,...,1,6.716595,0.000000,0.0,BR,0,1,0,0,0
16877,1,3,81067.0,0,0,0.0,0.0,0,0,BR,...,1,11.303044,0.000000,0.0,BR,0,1,0,0,0
16878,0,9,398372.0,0,0,0.0,0.0,0,0,BR,...,1,12.895144,0.000000,0.0,BR,0,1,0,0,0


In [22]:
base['var_K_100'] = (base['var_K'] * 100).round()

In [23]:
drop_col = ['var_C', 'var_F', 'var_G', 'var_J', 'pais_agg', 'var_K']
base = base.drop(columns=drop_col, errors='ignore')
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16880 entries, 0 to 16879
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   var_A              16880 non-null  int64  
 1   var_B              16880 non-null  int64  
 2   var_D              16880 non-null  int64  
 3   var_E              16880 non-null  int64  
 4   var_H              16880 non-null  int64  
 5   var_I              16880 non-null  int64  
 6   var_L              16880 non-null  int64  
 7   var_M              16880 non-null  int64  
 8   var_N              16880 non-null  int64  
 9   var_O              16880 non-null  int64  
 10  var_P              16880 non-null  int64  
 11  var_Q              16863 non-null  float64
 12  var_R              16877 non-null  float64
 13  var_S              15575 non-null  float64
 14  Monto              16413 non-null  float64
 15  Fraude             16880 non-null  int64  
 16  var_C_log          136

In [24]:
df = base[base['Monto'].notna()]

## Treino, Teste e Validação

In [25]:
rf_fs = ['var_S',
 'Monto',
 'var_C_log',
 'var_B',
 'pais_agg_ES_US_UY',
 'var_K_100',
 'var_P',
 'var_M',
 'var_L',
 'var_A',
 'var_E',
 'pais_agg_MX',
 'var_F_log',
 'pais_agg_AR',
 'pais_agg_BR',
 'var_Q',
 'var_D']

In [26]:
X = df[rf_fs]
y = df["Fraude"]

In [27]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=13)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=13)

In [28]:
X_train.shape, X_test.shape, X_val.shape

((11489, 17), (2462, 17), (2462, 17))

## Monitormento de Variáveis utilizando o PSI

In [29]:
psi_teste = {var: calculate_psi(X_train[var].dropna(), X_test[var].dropna()) for var in rf_fs}
psi_val = {var: calculate_psi(X_train[var].dropna(), X_val[var].dropna()) for var in rf_fs}

In [30]:
psi_df = pd.DataFrame({'Teste': psi_teste, 'Validacao': psi_val})
psi_df.sort_values('Teste', ascending=False)

,Teste,Validacao
var_L,0.0234,0.0048
pais_agg_BR,0.0110,0.0005
var_F_log,0.0075,0.0171
pais_agg_AR,0.0067,0.0015
pais_agg_ES_US_UY,0.0050,0.0000
var_E,0.0041,0.0015
var_M,0.0038,0.0071
var_B,0.0024,0.0026
var_A,0.0022,0.0052
var_C_log,0.0020,0.0023


## Monitoramento de Falsos Positivos

In [31]:
best = {'colsample_bytree': np.float64(0.9536957201740883),
 'gamma': np.float64(1.0416424955881598),
 'learning_rate': np.float64(0.047958206007245714),
 'max_depth': np.float64(9.0),
 'min_child_weight': np.float64(9.0),
 'n_estimators': np.float64(290.0),
 'reg_alpha': np.float64(3.855225280461912),
 'reg_lambda': np.float64(0.6684995061329557),
 'subsample': np.float64(0.8285019837004117)}

In [32]:
final_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=13,
    n_jobs=-1,
    max_depth=int(best['max_depth']),
    learning_rate=best['learning_rate'],
    n_estimators=int(best['n_estimators']),
    gamma=best['gamma'],
    min_child_weight=int(best['min_child_weight']),
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda']
)

final_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=np.float64(0.9536957201740883), device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None,
              gamma=np.float64(1.0416424955881598), grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=np.float64(0.047958206007245714), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=9, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=290, n_jobs=-1,
              num_parallel_tree=None, random_state=13, ...)

In [33]:
y_probs_test = final_model.predict_proba(X_test)[:, 1]
y_probs_val = final_model.predict_proba(X_val)[:, 1]

In [34]:
best_threshold_f1 = 0.33

In [35]:
y_pred_test = (y_probs_test > best_threshold_f1).astype(int)
y_pred_val = (y_probs_val > best_threshold_f1).astype(int)

In [36]:
analisar_metricas_por_decil(y_val, y_probs_val)

,Decil,Score Mínimo,Score Máximo,Transações,Fraudes,Hit Rate,Detection Rate,Alert Rate
0,10,639,984,247,199,0.8057,0.2953,0.1003
1,9,449,638,246,145,0.5894,0.2151,0.0999
2,8,331,448,246,91,0.3699,0.1350,0.0999
3,7,251,331,246,72,0.2927,0.1068,0.0999
4,6,178,251,246,57,0.2317,0.0846,0.0999
5,5,122,177,246,47,0.1911,0.0697,0.0999
6,4,91,122,246,27,0.1098,0.0401,0.0999
7,3,65,90,246,15,0.0610,0.0223,0.0999
8,2,40,64,246,15,0.0610,0.0223,0.0999
9,1,11,40,247,6,0.0243,0.0089,0.1003


In [37]:

decil_10_df = analisar_metricas_por_decil(y_val, y_probs_val)
decil_10_min_score = decil_10_df.iloc[0]['Score Mínimo'] / 1000
decil_10_max_score = decil_10_df.iloc[0]['Score Máximo'] / 1000

In [38]:
df_val = pd.concat([X_val, y_val], axis=1)
df_val['Fraude_prob'] = y_probs_val
df_val['Fraude_pred'] = y_pred_val
df_val

,var_S,Monto,var_C_log,var_B,pais_agg_ES_US_UY,var_K_100,var_P,var_M,var_L,var_A,var_E,pais_agg_MX,var_F_log,pais_agg_AR,pais_agg_BR,var_Q,var_D,Fraude,Fraude_prob,Fraude_pred
9500,13.94,35.85,NaN,8,0,NaN,1,2,0,0,0,0,0.000000,1,0,0.00,0,0,0.113220,0
15330,NaN,2.75,NaN,1,0,NaN,5,3,0,0,2,0,1.609438,0,1,0.00,2,0,0.067830,0
2949,50.50,132.66,6.381816,4,0,73.0,5,3,3,1,0,0,0.000000,1,0,459.57,0,1,0.896140,1
563,0.37,15.65,9.959301,12,1,79.0,4,4,2,0,3,0,0.223144,0,0,18.74,0,1,0.965301,1
128,50.49,145.98,7.975221,13,1,NaN,1,1,0,2,0,0,0.000000,0,0,0.00,0,1,0.977190,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3038,82.86,93.07,7.618251,4,0,67.0,1,1,0,1,1,0,0.039221,1,0,0.00,0,1,0.448597,1
12981,8.91,63.46,NaN,11,0,NaN,1,1,0,0,0,0,0.000000,0,1,0.00,0,0,0.080245,0
12365,11.60,17.47,NaN,6,0,NaN,2,2,0,0,0,1,0.000000,0,0,0.00,0,0,0.024890,0
1579,23.32,77.39,8.683385,8,0,NaN,1,2,0,0,0,0,0.000000,1,0,0.00,0,1,0.300091,0


In [39]:
D10_NFraude = df_val[(df_val['Fraude'] == 0) & (df_val['Fraude_prob'] >= decil_10_min_score) & (df_val['Fraude_prob'] <= decil_10_max_score)]
D10_NFraude

,var_S,Monto,var_C_log,var_B,pais_agg_ES_US_UY,var_K_100,var_P,var_M,var_L,var_A,var_E,pais_agg_MX,var_F_log,pais_agg_AR,pais_agg_BR,var_Q,var_D,Fraude,Fraude_prob,Fraude_pred
15837,87.09,27.17,9.789535,18,0,NaN,1,3,0,0,0,0,0.000000,0,1,0.00,0,0,0.766545,1
12514,87.40,103.88,13.036130,9,0,NaN,1,1,1,0,3,0,1.386294,0,1,0.00,3,0,0.733731,1
9041,91.58,104.21,0.000000,9,0,NaN,1,2,1,0,0,0,0.000000,1,0,0.00,0,0,0.796357,1
14850,51.45,47.76,6.432940,17,0,77.0,3,3,0,0,0,0,0.000000,0,1,0.00,0,0,0.714029,1
3139,NaN,55.98,11.182364,15,0,54.0,1,1,0,0,0,0,0.000000,1,0,0.00,0,0,0.666691,1
9455,15.99,43.37,9.336709,9,0,NaN,3,2,0,0,0,0,0.000000,1,0,0.00,0,0,0.642507,1
12440,17.62,331.64,NaN,9,1,72.0,1,2,2,4,0,0,0.000000,0,0,0.00,0,0,0.963240,1
11075,44.78,29.78,0.000000,13,0,58.0,1,2,0,0,0,0,0.000000,0,0,0.00,0,0,0.646433,1
4942,10.91,40.50,9.962275,9,0,58.0,1,1,0,1,0,0,0.000000,1,0,0.00,0,0,0.642338,1
3352,98.51,125.80,12.699210,14,0,NaN,1,2,0,0,16,0,0.000000,1,0,0.00,0,0,0.700854,1
